In [ ]:
from databricks.sdk.runtime import spark, dbutils
import json
import os

In [ ]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
if not notebook_dir.startswith("/Workspace"):
    notebook_dir = "/Workspace" + notebook_dir

with open(os.path.join(notebook_dir, "tables_config.json"), "r") as f:
    config = json.load(f)

CATALOG_SCHEMA = "lh_nautical.bronze"
CONTROL_TABLE  = f"{CATALOG_SCHEMA}._ingestion_control"
AUDIT_TABLE    = f"{CATALOG_SCHEMA}._ingestion_audit"

tables = [t["name"] for t in config["tables"]]

In [ ]:
# Drop all ingested data tables
for table in tables:
    full_name = f"{CATALOG_SCHEMA}.{table}"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    print(f"Dropped: {full_name}")

# Truncate control table so incremental watermarks reset
spark.sql(f"DELETE FROM {CONTROL_TABLE} WHERE 1=1")
print(f"Cleared: {CONTROL_TABLE}")

print("Done.")